# 17 高级分箱 + 完整可视化

覆盖 GeneticBinning / KernelDensityBinning / ORBinning / GBMEncoder / CatBoostEncoder /
risk_plots / model_plots / bin_trend_plot / overdues_bin_plot / batch_bin_trend_plot。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from hscredit import (germancredit, init_setting, seed_everything,
    GeneticBinning, KernelDensityBinning, ORBinning, OptimalBinning,
    WOEEncoder, LogisticRegression,
)
from hscredit.core.metrics import ks, auc
seed_everything(42); init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print('准备完成')

## 1. GeneticBinning — 遗传算法分箱

In [ ]:
try:
    gen_b = GeneticBinning(max_n_bins=5, n_population=30, n_generations=20, random_state=42)
    gen_b.fit(X_tr[['credit.amount']], y_tr)
    bt = gen_b.bin_tables_['credit.amount']
    print('遗传分箱 IV:', round(bt['分档IV值'].sum(), 4))
    print(bt[['分箱标签','样本总数','坏样本率','分档IV值']])
except Exception as e:
    print(f'GeneticBinning: {e}')

## 2. KernelDensityBinning — 核密度分箱

In [ ]:
try:
    kd_b = KernelDensityBinning(max_n_bins=5, bandwidth='silverman')
    kd_b.fit(X_tr[['credit.amount']], y_tr)
    bt_kd = kd_b.bin_tables_['credit.amount']
    print('核密度分箱 IV:', round(bt_kd['分档IV值'].sum(), 4))
    print(bt_kd[['分箱标签','样本总数','坏样本率']])
except Exception as e:
    print(f'KernelDensityBinning: {e}')

## 3. ORBinning — OR-Tools整数规划最优分箱

In [ ]:
try:
    or_b = ORBinning(max_n_bins=5, monotonic='auto')
    or_b.fit(X_tr[['credit.amount']], y_tr)
    bt_or = or_b.bin_tables_['credit.amount']
    print('OR分箱 IV:', round(bt_or['分档IV值'].sum(), 4))
    print(bt_or[['分箱标签','样本总数','坏样本率','分档IV值']])
except Exception as e:
    print(f'ORBinning: {e}')

## 4. GBMEncoder / CatBoostEncoder

In [ ]:
from hscredit import GBMEncoder
try:
    gbm_enc = GBMEncoder(n_estimators=50, random_state=42)
    gbm_enc.fit(X_tr, y_tr)
    X_gbm = gbm_enc.transform(X_te)
    print('GBMEncoder 输出形状:', X_gbm.shape)
    print(X_gbm.head(3))
except Exception as e:
    print(f'GBMEncoder: {e}')

In [ ]:
from hscredit import CatBoostEncoder
try:
    cbe = CatBoostEncoder()
    cbe.fit(X_tr, y_tr)
    X_cbe = cbe.transform(X_te)
    print('CatBoostEncoder:', X_cbe.head(3))
except Exception as e:
    print(f'CatBoostEncoder: {e}')

## 5. bin_trend_plot — 特征分箱风险趋势图

In [ ]:
from hscredit import bin_trend_plot
# 模拟时间维度
np.random.seed(42)
start = pd.Timestamp('2022-01-01')
df_ts = df.copy()
df_ts['apply_date'] = pd.to_datetime([start + pd.Timedelta(days=int(d)) for d in np.random.randint(0,730,len(df_ts))])
df_ts['channel'] = np.random.choice(['线上','线下','中介'], len(df_ts))

# 按时间维度展示分箱风险趋势
fig = bin_trend_plot(df_ts, feature='credit.amount', target=target,
    date_col='apply_date', date_freq='Q', n_bins=5)
plt.show()

In [ ]:
# 按渠道维度展示
fig = bin_trend_plot(df_ts, feature='duration.in.month', target=target,
    dimension_cols='channel', n_bins=5)
plt.show()

## 6. batch_bin_trend_plot — 批量趋势图

In [ ]:
from hscredit import batch_bin_trend_plot
figs = batch_bin_trend_plot(df_ts, features=num_feats[:3], target=target,
    date_col='apply_date', date_freq='Q', max_features=3, sort_by='iv')
for feat, fig in figs.items():
    print(f'特征: {feat}')
    plt.show()

## 7. overdues_bin_plot — 多逾期定义对比

In [ ]:
from hscredit import overdues_bin_plot
np.random.seed(42)
df_od = df.copy()
df_od['fpd15'] = np.random.binomial(1, 0.1, len(df_od))
df_od['fpd30'] = np.random.binomial(1, 0.15, len(df_od))
df_od['fpd60'] = np.random.binomial(1, 0.07, len(df_od))
fig = overdues_bin_plot(df_od, feature='credit.amount',
    targets=['fpd15','fpd30','fpd60'], n_bins=6)
plt.show()

## 8. risk_plots — 风控专用图表

In [ ]:
from hscredit import (threshold_analysis_plot, strategy_compare_plot,
    vintage_plot, feature_importance_plot,
    approval_rate_trend_plot, bad_rate_trend_plot)
woe = WOEEncoder(max_n_bins=5); X_woe = woe.fit_transform(df[num_feats], y)
lr = LogisticRegression(max_iter=1000); lr.fit(X_woe, y)
prob = lr.predict_proba(X_woe)[:,1]

# 阈值分析
fig = threshold_analysis_plot(y.values, prob, thresholds=np.linspace(0.1,0.9,17))
plt.show()

In [ ]:
# 策略对比（两个不同阈值）
fig = strategy_compare_plot(y.values, prob,
    strategies={'宽松(0.2)': 0.2, '严格(0.35)': 0.35, '默认(0.3)': 0.3})
plt.show()

In [ ]:
# Vintage图
np.random.seed(42)
df_vint = pd.DataFrame({
    'cohort': np.random.choice(['2022Q1','2022Q2','2022Q3','2022Q4'], 500),
    'mob': np.random.randint(1, 13, 500),
    'is_bad': np.random.binomial(1, 0.1, 500),
})
fig = vintage_plot(df_vint, cohort_col='cohort', mob_col='mob', target='is_bad')
plt.show()

In [ ]:
# 特征重要性图
from sklearn.ensemble import RandomForestClassifier
rf_raw = RandomForestClassifier(n_estimators=50, random_state=42)
rf_raw.fit(df[num_feats], y)
importances = pd.Series(rf_raw.feature_importances_, index=num_feats)
fig = feature_importance_plot(importances, top_n=10)
plt.show()

In [ ]:
# 通过率&坏率趋势
df_score = df_ts.copy()
df_score['score'] = lr.predict_proba(woe.transform(df_score[num_feats]))[:,1]
df_score['approved'] = (df_score['score'] < 0.3).astype(int)
fig = approval_rate_trend_plot(df_score, date_col='apply_date', decision_col='approved', freq='Q')
plt.show()
fig = bad_rate_trend_plot(df_score, date_col='apply_date', target=target, freq='Q')
plt.show()